# Finetuning Recipes — Steps 01–02 on Google Colab GPU

This notebook runs the first two stages of the repository on Colab:

1. **Step 01:** Download arXiv papers and build CPT JSONL datasets.
2. **Step 02:** Continued pre-training / CPT with GPU via Unsloth when available.

Before running: in Colab, go to **Runtime → Change runtime type → T4 GPU** or better.


## 0. Check GPU


In [ ]:
!nvidia-smi


## 1. Get the repository

If your repo is on GitHub, set `REPO_URL` and run this cell. If you uploaded the repo manually to Colab, set `PROJECT_DIR` to that folder instead.


In [ ]:
from pathlib import Path

# Change this if your repo is pushed somewhere else.
REPO_URL = "https://github.com/YOUR_USERNAME/finetuning_recipes.git"
PROJECT_DIR = Path("/content/finetuning_recipes")

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository URL, or upload the repo to /content/finetuning_recipes.")
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!pwd
!ls


## 2. Install dependencies

This uses `uv` and the repository's `pyproject.toml`. The first run can take several minutes.


In [ ]:
!pip install -q uv
!uv sync


## 3. Configure run parameters

Start small to verify the pipeline. Increase `TARGET_COUNT`, `EPOCHS`, and batch size later.


In [ ]:
from pathlib import Path

PAPERS_DIR = Path("step_01_data_prep/cpt/papers")
TARGET_COUNT = 100       # Use 20 for a smoke test, 500+ for a better CPT run
WORKERS = 5
TOPIC_SET = "ml"       # "ml" or "cs-ai"

BASE_MODEL_ID = "HuggingFaceTB/SmolLM-135M"
OUTPUT_MODEL_ID = "smollm_arxiv_cpt_colab"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 8           # T4 may need 4 or 8; A100 can use more
EPOCHS = 1               # Increase after the smoke test

PAPERS_DIR.mkdir(parents=True, exist_ok=True)


## 4. Step 01 — Download arXiv papers


In [ ]:
!uv run python step_01_data_prep/cpt/download_arxiv_batch.py --output-dir {PAPERS_DIR} --target-count {TARGET_COUNT} --workers {WORKERS} --topic-set {TOPIC_SET}


In [ ]:
txt_count = len(list(PAPERS_DIR.rglob("*.txt")))
print(f"Downloaded text files: {txt_count}")


## 5. Step 01 — Build CPT train/validation JSONL files


In [ ]:
!uv run python step_01_data_prep/cpt/get_dataset.py {PAPERS_DIR}
!ls -lh *.jsonl


In [ ]:
import glob

train_files = sorted(glob.glob("cpt_train_dataset_*.jsonl"))
val_files = sorted(glob.glob("cpt_val_dataset_*.jsonl"))

if not train_files or not val_files:
    raise FileNotFoundError("Could not find generated cpt_train_dataset_*.jsonl and cpt_val_dataset_*.jsonl files.")

TRAIN_JSONL = train_files[-1]
VAL_JSONL = val_files[-1]
print("TRAIN_JSONL =", TRAIN_JSONL)
print("VAL_JSONL   =", VAL_JSONL)


## 6. Step 02 — Continued pre-training on GPU

On Colab with CUDA, the script should use Unsloth automatically. If you get out-of-memory errors, lower `BATCH_SIZE` to `4` or `2`.


In [ ]:
!uv run python step_02_cpt/sft.py --base_model_id {BASE_MODEL_ID} --output_model_id {OUTPUT_MODEL_ID} --dataset_path {TRAIN_JSONL} --test_dataset_path {VAL_JSONL} --max_seq_length {MAX_SEQ_LENGTH} --batch_size {BATCH_SIZE} --epochs {EPOCHS}


## 7. Check saved model


In [ ]:
!find models -maxdepth 3 -type f | head -50


## 8. Optional — Save model/datasets to Google Drive

Run this if you want the trained adapter and generated datasets to persist after the Colab runtime shuts down.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/finetuning_recipes_outputs
# !cp -r models/{OUTPUT_MODEL_ID} /content/drive/MyDrive/finetuning_recipes_outputs/
# !cp cpt_train_dataset_*.jsonl cpt_val_dataset_*.jsonl /content/drive/MyDrive/finetuning_recipes_outputs/
